# Reproduce the CisFalcon flagship number in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/belumume/cisfalcon/blob/main/reproduce_flagship.ipynb)

One click, no install, no GPU, no API key. This fetches the committed per-design cross-lab scores (93,435 AI-designed enhancers from an independent lab, Gosai/Tewhey 2024) straight from GitHub and re-derives the headline numbers with a plain rank-sum AUROC in pure numpy, so you can watch **AUROC 0.8013** print rather than take it on faith. Full method and every caveat: `PREREG.md`.

In [ ]:
# CisFalcon flagship, reproduced from the committed cross-lab scores.
import io, urllib.request, csv
import numpy as np

URL = "https://raw.githubusercontent.com/belumume/cisfalcon/main/data/gosai_designed/designed_scored.csv"
raw = urllib.request.urlopen(urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})).read().decode()
rows = list(csv.DictReader(io.StringIO(raw)))
y = np.array([int(r["measured_fail"]) for r in rows])       # 1 = wet-lab specificity failure
gap = np.array([float(r["pred_gap"]) for r in rows])        # CisFalcon predicted specificity gap
n = len(y); base = y.mean()

def auroc(score, labels):
    """Rank-sum (Mann-Whitney) AUROC; higher score = higher predicted failure risk."""
    order = np.argsort(score, kind="mergesort")
    rank = np.empty(len(score)); rank[order] = np.arange(1, len(score) + 1)
    n1 = labels.sum(); n0 = len(labels) - n1
    return float((rank[labels == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

auc = auroc(-gap, y)
order = np.argsort(gap)                                     # riskiest (smallest gap) first
ys = y[order]
k2 = int(round(0.02 * n))
ppv2 = ys[:k2].mean()
safe_half = ys[n // 2:].mean()
rng = np.random.default_rng(0)
auc_random = auroc(rng.random(n), y)

print(f"designs (independent lab, zero training overlap): {n:,}")
print(f"wet-lab specificity-failure base rate           : {base*100:.2f}%  (~1 in {1/base:.0f})")
print(f"cross-lab AUROC                                 : {auc:.4f}")
print(f"  random-score baseline                         : {auc_random:.4f}")
print(f"riskiest 2% flagged, truly fail                 : {ppv2*100:.1f}%  ({ppv2/base:.2f}x base)")
print(f"synthesize the safest half, failure rate        : {safe_half*100:.2f}%  ({(1-safe_half/base)*100:.0f}% fewer)")
print()
print("These are the deployment numbers for a mixed batch. The fully-conditioned per-sequence")
print("signal (cell and generator base-rates removed) is 0.66; see cell_prior_baseline.py and PREREG.md.")
